# NB9 — Detection Colab smoke test

Run RF-DETR → crops → FashionCLIP Core-7 → scorer handoff on a real outfit image. This notebook is intentionally tied to `fix/detection-colab-smoke` so experiments stay isolated from the teammate branch.


In [ ]:
from pathlib import Path
import importlib, os, subprocess, sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "fix/detection-colab-smoke"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_ROOT)
importlib.invalidate_caches()
for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        sys.modules.pop(name, None)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-detection.txt")], check=True)
HEAD = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
CURRENT_BRANCH = subprocess.check_output(["git", "-C", str(REPO_ROOT), "branch", "--show-current"], text=True).strip()
print("Branch:", CURRENT_BRANCH)
print("HEAD  :", HEAD)


In [ ]:
test_run = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_detection_core7.py", "-v"], cwd=REPO_ROOT, text=True, capture_output=True)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode != 0:
    raise RuntimeError("Detection contract tests failed")

import torch
assert torch.cuda.is_available(), "Select Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))


## Image
By default this uses the committed sample at `tests/animage.jpg`. Set `UPLOAD_CUSTOM_IMAGE=True` to test your own image.


In [ ]:
UPLOAD_CUSTOM_IMAGE = False
if UPLOAD_CUSTOM_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No image uploaded")
    IMAGE_PATH = (Path.cwd() / next(iter(uploaded))).resolve()
else:
    IMAGE_PATH = REPO_ROOT / "tests" / "animage.jpg"
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(IMAGE_PATH)
print("IMAGE_PATH:", IMAGE_PATH)


In [ ]:
from src.detection import DetectionPipeline, load_detection_config
CONFIG_PATH = REPO_ROOT / "configs" / "detection_rfdetr_fashionclip_core7_v1.json"
config = load_detection_config(CONFIG_PATH)
pipeline = DetectionPipeline(config, device="cuda")
result, image = pipeline.run(IMAGE_PATH)
print("accepted garments  :", len(result.garments))
print("rejected detections:", len(result.rejected_detections))
print("RF-DETR runtime ms :", result.detector_runtime_ms)
for i, g in enumerate(result.garments):
    print(i, g.candidate.detector_label, "->", g.category.coarse_category, f"det={g.candidate.detector_confidence}", f"sim={g.category.similarity:+.4f}", f"margin={g.category.margin:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(image)
for g in result.garments:
    x0, y0, x1, y1 = g.crop_box_xyxy
    ax.add_patch(patches.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, linewidth=2))
    ax.text(x0, max(0, y0-4), f"{g.candidate.detector_label} → {g.category.coarse_category}", fontsize=8)
ax.axis("off")
plt.show()

DETECTOR_LABEL_TO_CORE7 = {"shirt, blouse":"TOP", "top, t-shirt, sweatshirt":"TOP", "sweater":"TOP", "cardigan":"OUTERWEAR", "jacket":"OUTERWEAR", "vest":"OUTERWEAR", "pants":"BOTTOM", "shorts":"BOTTOM", "skirt":"BOTTOM", "coat":"OUTERWEAR", "dress":"DRESS", "jumpsuit":"DRESS", "cape":"OUTERWEAR", "hat":"HAT", "shoe":"SHOES", "bag, wallet":"BAG"}
agree = 0
for i, g in enumerate(result.garments):
    detector_core7 = DETECTOR_LABEL_TO_CORE7.get(g.candidate.detector_label)
    fashionclip_core7 = g.category.coarse_category
    same = detector_core7 == fashionclip_core7
    agree += int(same)
    norm = float(torch.linalg.vector_norm(g.embedding.float()).item())
    print(f"[{i}] detector→Core7={detector_core7} | FashionCLIP→Core7={fashionclip_core7} | agree={same} | norm={norm:.6f}")
if result.garments:
    print(f"Agreement: {agree}/{len(result.garments)} ({agree/len(result.garments):.1%})")


In [ ]:
from src.detection.pipeline import save_detection_result
RUN_DIR = REPO_ROOT / "outputs" / "detection_v1" / IMAGE_PATH.stem
saved = save_detection_result(result, image, RUN_DIR, scorer_min_items=config.scorer_min_items, scorer_max_items=config.scorer_max_items)
print(saved)

if saved["scorer_inputs_path"] is not None:
    scorer_inputs = torch.load(saved["scorer_inputs_path"], map_location="cpu")
    for key, value in scorer_inputs.items():
        print(key, tuple(value.shape), value.dtype)
    norms = torch.linalg.vector_norm(scorer_inputs["item_embeddings"].float(), dim=-1)
    print("embedding norms:", norms)
else:
    print("SCORER HANDOFF SKIPPED:", saved["scorer_handoff_error"])


In [ ]:
if saved["scorer_inputs_path"] is not None:
    from src.scorer.checkpoint import load_checkpoint
    from src.scorer.model import TypeAwarePairwiseScorer
    BEST_PATH = REPO_ROOT / "artifacts" / "checkpoints" / "type_aware_pairwise_v1" / "final_val_auc_v5_seed42" / "best.pt"
    payload = load_checkpoint(BEST_PATH, map_location="cpu")
    scorer = TypeAwarePairwiseScorer.from_config(payload["config"])
    scorer.load_state_dict(payload["model_state_dict"])
    scorer.to("cuda").eval()
    batch = {k: v.to("cuda") for k, v in scorer_inputs.items()}
    with torch.inference_mode():
        out = scorer(batch["item_embeddings"], batch["coarse_category_ids"], batch["item_mask"])
    print("Frozen V5 compatibility_logit:", float(out["compatibility_logit"].item()))
    print("Raw uncalibrated logit — not a probability.")


## Read failures by layer
1. Wrong/missing boxes → RF-DETR/localization.
2. Boxes right but Core-7 wrong → category strategy.
3. Boxes + category right but scorer weird → investigate crop/domain shift.
